# 09 - resolution and field of view: what we feed the model today, and what we could

The loader rescales every slice to a fixed millimetres-per-pixel, then centre-crops or pads it to a fixed pixel size. Two numbers set that:

- `mm_per_px` - how many millimetres of real knee each pixel covers. Smaller = finer detail.
- `size` - the output side in pixels.

Their product is the **field of view**: how many millimetres of anatomy are in frame. Today that is `0.40 x 512 = 205 mm`.

The scanners in this dataset typically acquire **512 x 512 pixels over a 160 mm field of view**, i.e. **0.312 mm per pixel**. So on a typical study we currently shrink a native 512 px image down to about 400 px and then pad it back out to 512. We lose real detail and spend about a fifth of every image on padding.

This notebook shows that on real studies, side by side, so the tradeoff is visible rather than argued.

Run one cell at a time.

In [1]:
# Cell 1 - load the local shards and see what we actually have to work with.
# Shards store each series at its NATIVE pixel size, with the DICOM pixel spacing in the
# metadata, and the loader does the rescaling at training time. That is what makes this
# comparison honest: we can replay both settings from the same stored pixels.

import json
from pathlib import Path

import numpy as np
import pandas as pd

if "ROOT" not in globals():
    ROOT = Path.cwd().parent if Path.cwd().name == "data_code" else Path.cwd()
SHARD_DIR = ROOT / "data" / "shards"


def read_shard(path):
    """(meta dict, {series_uid: uint8 array of shape (n_slices, h, w)}) for one shard."""
    z = np.load(path, allow_pickle=True)
    meta = json.loads(bytes(z["meta"]).decode("utf-8"))
    vols = {k: z[k] for k in z.files if k != "meta"}
    return meta, vols


shard_files = sorted(SHARD_DIR.glob("*.npz"))
print(f"{len(shard_files)} shards locally under {SHARD_DIR}")

rows = []
for f in shard_files:
    meta, vols = read_shard(f)
    for uid, info in meta["series"].items():
        if uid not in vols:
            continue
        n, h, w = vols[uid].shape
        sp = (info.get("pixel_spacing") or [None, None])[1]
        rows.append({
            "shard": f.name,
            "study": meta["study"],
            "series": uid,
            "desc": info.get("series_desc", ""),
            "n_slices": n,
            "h": h, "w": w,
            "mm_per_px": sp,
            "fov_mm": None if sp is None else round(sp * w, 1),
        })

S = pd.DataFrame(rows)
print(f"{S.study.nunique()} studies, {len(S)} series\n")
print("native pixel spacing (mm per pixel):")
print(S.mm_per_px.describe().round(3).to_string())
print("\nnative field of view (mm across the image):")
print(S.fov_mm.describe().round(1).to_string())
print("\nWhat to look for: a median spacing near 0.31 and a median field of view near 160")
print("mean the scanner already gave us finer pixels than the 0.40 we resample to, over a")
print("narrower frame than the 205 mm we ask for.")
S.head(10)

5 shards locally under d:\My Projects\knee_abnormality_detection\data\shards
5 studies, 31 series

native pixel spacing (mm per pixel):
count    31.000
mean      0.319
std       0.104
min       0.137
25%       0.250
50%       0.299
75%       0.417
max       0.625

native field of view (mm across the image):
count     31.0
mean     159.1
std       16.1
min      100.0
25%      150.0
50%      160.0
75%      169.0
max      180.0

What to look for: a median spacing near 0.31 and a median field of view near 160
mean the scanner already gave us finer pixels than the 0.40 we resample to, over a
narrower frame than the 205 mm we ask for.


,shard,study,series,desc,n_slices,h,w,mm_per_px,fov_mm
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,DP SPIR CS_SAG,22,512,512,0.330078,169.0
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,STIR CS_ AX,18,560,560,0.287712,161.1
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,T1_TSE_CS,16,576,576,0.298910,172.2
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,DP SPAIR_CS,16,560,560,0.307450,172.2
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,DP CS_SAG,22,640,640,0.264062,169.0
5,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.12689544351656485342...,pd_tse_fs_cor_416_p2_s2,38,416,416,0.432692,180.0
6,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.26515132645256697953...,pd_tse_fs_tra_416_p2_s2,48,416,416,0.432692,180.0
7,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.30827030879314217305...,t1_tse_cor_448_p2_s2,38,416,416,0.432692,180.0
8,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.33836648034724869721...,t2_tse_sag_448_2.5mm,38,416,416,0.432692,180.0
9,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.10009278692606631573...,1.2.826.0.1.3680043.8.498.93288538765996574291...,pd_tse_fs_sag_416_p2_s2,38,416,416,0.432692,180.0


## The two pipelines, side by side`KneeShardDataset` does exactly two geometric things to every slice:1. **resample** by `native_spacing / mm_per_px`, so a knee is the same pixel size on every scanner2. **`_fit`**: centre-crop anything larger than `size`, zero-pad anything smallerThe next cell reimplements both, byte-for-byte, so what you see is what the model sees.

In [ ]:
# Cell 2 - the two geometric steps, copied from KneeShardDataset so nothing is approximated.import cv2CURRENT  = dict(name="current",  mm_per_px=0.40, size=512)   # 205 mm of framePROPOSED = dict(name="proposed", mm_per_px=0.27, size=512)   # 138 mm, same tensorfor s in (CURRENT, PROPOSED):    s["fov_mm"] = s["mm_per_px"] * s["size"]def resample_and_fit(vol, native_mm_per_px, mm_per_px, size):    """Exactly what the dataset does: rescale to a common physical scale, then centre    crop or pad to size x size. Returns (fitted volume, scaled size before fitting)."""    v = vol    if mm_per_px:        f = float(native_mm_per_px) / mm_per_px        if abs(f - 1) > 0.02:                      # the dataset skips a resize within 2%            h, w = int(round(v.shape[1] * f)), int(round(v.shape[2] * f))            v = np.stack([cv2.resize(s, (w, h), interpolation=cv2.INTER_AREA) for s in v])    scaled = v.shape[1:]    # _fit: crop the excess, pad the shortfall, both centred    s_, (h, w) = size, v.shape[1:]    if h > s_:        t = (h - s_) // 2; v = v[:, t:t + s_]    if w > s_:        l = (w - s_) // 2; v = v[:, :, l:l + s_]    h, w = v.shape[1:]    if h < s_ or w < s_:        ph, pw = max(0, s_ - h), max(0, s_ - w)        v = np.pad(v, ((0, 0), (ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)))    return v, scaledprint("%-10s %-11s %-8s %s" % ("setting", "mm_per_px", "size", "field of view"))for s in (CURRENT, PROPOSED):    print("%-10s %-11.2f %-8d %.0f mm" % (s["name"], s["mm_per_px"], s["size"], s["fov_mm"]))print("\nA typical study here is 512 px over 160 mm, i.e. 0.312 mm/px, so:")for s in (CURRENT, PROPOSED):    f = 0.312 / s["mm_per_px"]    print("  %-9s native 512 px -> %.0f px, then %s to %d"          % (s["name"], 512 * f, "cropped" if 512 * f > s["size"] else "PADDED", s["size"]))

## What the model actually seesRed dashed box = the current 205 mm frame. Green = the proposed 138 mm frame.Grey border = zero padding, i.e. pixels that carry no information at all.

In [ ]:
# Cell 3 - pick random studies and show current vs proposed on the same slice.import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesif "RNG_SEED" not in globals():    RNG_SEED = 0rng = np.random.default_rng(RNG_SEED)          # change RNG_SEED and rerun for new studiesusable = S[S.mm_per_px.notna() & (S.n_slices >= 8)]picks = usable.sample(min(4, len(usable)), random_state=RNG_SEED)fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4.3 * len(picks)))axes = np.atleast_2d(axes)for r, (_, row) in enumerate(picks.iterrows()):    _, vols = read_shard(SHARD_DIR / row["shard"])    vol = vols[row["series"]]    mid = vol.shape[0] // 2    one = vol[mid:mid + 1].astype(np.float32)    # native, for reference    axes[r, 0].imshow(one[0], cmap="gray")    axes[r, 0].set_title("native %dx%d @ %.3f mm/px\n= %.0f mm of knee"                         % (row["w"], row["h"], row["mm_per_px"], row["fov_mm"]),                         fontsize=9)    for c, setting in enumerate((CURRENT, PROPOSED), start=1):        fitted, scaled = resample_and_fit(one, row["mm_per_px"],                                          setting["mm_per_px"], setting["size"])        axes[r, c].imshow(fitted[0], cmap="gray")        pad_frac = 1.0 - (min(scaled[0], setting["size"]) *                          min(scaled[1], setting["size"])) / setting["size"] ** 2        axes[r, c].set_title("%s: %.2f mm/px, %d px = %.0f mm\n"                             "knee occupies %dx%d, padding %.0f%%"                             % (setting["name"], setting["mm_per_px"], setting["size"],                                setting["fov_mm"], min(scaled[1], setting["size"]),                                min(scaled[0], setting["size"]), 100 * pad_frac),                             fontsize=9)        # outline the region the OTHER setting would keep, in this one's pixels        other = PROPOSED if setting is CURRENT else CURRENT        side_px = other["fov_mm"] / setting["mm_per_px"]        o = (setting["size"] - side_px) / 2        axes[r, c].add_patch(mpatches.Rectangle(            (o, o), side_px, side_px, fill=False, lw=1.6, ls="--",            ec="lime" if other is PROPOSED else "red"))    for a in axes[r]:        a.set_xticks([]); a.set_yticks([])fig.suptitle("dashed box = the other setting's field of view", y=1.001, fontsize=10)fig.tight_layout()plt.show()

## How much of the frame is wasted, across the whole local sampleThe visual is one slice. This is the arithmetic over every series we have locally.

In [ ]:
# Cell 4 - padding and pixels-on-knee, for every local series.rows = []for _, row in usable.iterrows():    for setting in (CURRENT, PROPOSED):        f = row["mm_per_px"] / setting["mm_per_px"]        scaled = row["w"] * f        kept = min(scaled, setting["size"])        rows.append({            "setting": setting["name"],            "series": row["series"],            # fraction of the OUTPUT tensor that is zero padding            "padding_pct": 100 * max(0.0, 1 - (kept / setting["size"]) ** 2),            # how many output pixels land on real anatomy            "knee_px": kept ** 2,            # how much real knee is inside the frame, in mm            "frame_mm": setting["fov_mm"],            "anatomy_mm": min(row["fov_mm"], setting["fov_mm"]),            "cropped_mm": max(0.0, row["fov_mm"] - setting["fov_mm"]),        })D = pd.DataFrame(rows)print(D.groupby("setting")[["padding_pct", "knee_px", "anatomy_mm", "cropped_mm"]]      .median().round(1).to_string())print()g = D.groupby("setting").knee_px.median()print("median output pixels landing on anatomy: current %.0f, proposed %.0f  (%.2fx)"      % (g["current"], g["proposed"], g["proposed"] / g["current"]))print("median mm of knee thrown away by the crop: current %.0f, proposed %.0f"      % (D[D.setting == "current"].cropped_mm.median(),         D[D.setting == "proposed"].cropped_mm.median()))print()print("Read it this way: the proposed setting keeps a SMALLER piece of the patient but")print("spends far more of the 512x512 tensor on it. The question the run answers is")print("whether the discarded outer ring held anything the model was using.")